In [4]:
#bài 3
import copy
import math
import numpy

# Các hằng số
X = "X"
O = "O"
EMPTY = None

# Cấu hình trò chơi NxN
N = 5
WIN_LEN = 4
MAX_DEPTH = 3

def initial_state(n):
    return [[EMPTY for _ in range(n)] for _ in range(n)]

def player(board):
    count = 0
    for row in board:
        for cell in row:
            if cell: count += 1
    return O if count % 2 != 0 else X

def actions(board):
    """
    Tối ưu: Chỉ xét các ô trống nằm cạnh các ô đã có quân (trong bán kính 1 ô)
    để giảm số lượng nhánh phải duyệt cho ma trận NxN.
    """
    res = set()
    n = len(board)
    has_piece = False

    for i in range(n):
        for j in range(n):
            if board[i][j] == EMPTY:
                # Kiểm tra xem xung quanh có quân nào không
                is_near = False
                for di in range(-1, 2):
                    for dj in range(-1, 2):
                        if 0 <= i + di < n and 0 <= j + dj < n and board[i+di][j+dj] is not None:
                            is_near = True
                            break
                if is_near:
                    res.add((i, j))
                has_piece = has_piece or (board[i][j] is not None)

    # Nếu bàn cờ trống, đánh vào giữa
    if not res:
        return {(n // 2, n // 2)}
    return res

def result(board, action):
    curr_player = player(board)
    result_board = copy.deepcopy(board)
    (i, j) = action
    result_board[i][j] = curr_player
    return result_board

def winner(board):
    n = len(board)
    for r in range(n):
        for c in range(n):
            if board[r][c] == EMPTY: continue
            curr = board[r][c]
            # Kiểm tra 4 hướng: ngang, dọc, chéo xuôi, chéo ngược
            for dr, dc in [(0, 1), (1, 0), (1, 1), (1, -1)]:
                count = 0
                for i in range(WIN_LEN):
                    nr, nc = r + dr * i, c + dc * i
                    if 0 <= nr < n and 0 <= nc < n and board[nr][nc] == curr:
                        count += 1
                    else: break
                if count == WIN_LEN: return curr
    return None

def terminal(board):
    if winner(board) is not None:
        return True
    return all(cell != EMPTY for row in board for cell in row)

def evaluate(board):
    """
    Hàm Heuristic (Lượng giá): Đánh giá điểm cho AI khi chưa kết thúc ván đấu.
    Điểm cộng cho AI (X), điểm trừ cho người chơi (O).
    """
    res = winner(board)
    if res == X: return 1000
    if res == O: return -1000

    # Nếu chưa thắng, đếm số chuỗi quân cờ tiềm năng (đơn giản hóa)
    # Ở đây chúng ta có thể viết thêm logic chấm điểm dựa trên chuỗi 2, 3 quân.
    return 0

def maxValue(state, depth):
    if terminal(state) or depth == 0:
        return evaluate(state)
    v = -math.inf
    for action in actions(state):
        v = max(v, minValue(result(state, action), depth - 1))
    return v

def minValue(state, depth):
    if terminal(state) or depth == 0:
        return evaluate(state)
    v = math.inf
    for action in actions(state):
        v = min(v, maxValue(result(state, action), depth - 1))
    return v

def minimax(board):
    curr_player = player(board)
    best_move = None

    if curr_player == X:
        v = -math.inf
        for action in actions(board):
            score = minValue(result(board, action), MAX_DEPTH)
            if score > v:
                v = score
                best_move = action
    else:
        v = math.inf
        for action in actions(board):
            score = maxValue(result(board, action), MAX_DEPTH)
            if score < v:
                v = score
                best_move = action
    return best_move

# --- CHƯƠNG TRÌNH CHÍNH ---
if __name__ == "__main__":
    board = initial_state(N)
    print(f"TicTacToe {N}x{N} - Cần {WIN_LEN} quân để thắng")
    user_symbol = input("Chọn quân (X/O): ").upper()
    ai_symbol = O if user_symbol == X else X

    while True:
        print(numpy.array(board))
        if terminal(board):
            res = winner(board)
            print("KẾT THÚC: " + (f"{res} thắng!" if res else "Hòa!"))
            break

        curr_p = player(board)
        if curr_p == user_symbol:
            print(f"Lượt của bạn ({user_symbol})")
            r = int(input("Dòng: "))
            c = int(input("Cột: "))
            if board[r][c] == EMPTY:
                board = result(board, (r, c))
            else:
                print("Ô đã có quân!")
        else:
            print(f"AI ({ai_symbol}) đang tính toán...")
            move = minimax(board)
            board = result(board, move)

TicTacToe 5x5 - Cần 4 quân để thắng
Chọn quân (X/O): X
[[None None None None None]
 [None None None None None]
 [None None None None None]
 [None None None None None]
 [None None None None None]]
Lượt của bạn (X)
Dòng: 2
Cột: 2
[[None None None None None]
 [None None None None None]
 [None None 'X' None None]
 [None None None None None]
 [None None None None None]]
AI (O) đang tính toán...
[[None None None None None]
 [None None 'O' None None]
 [None None 'X' None None]
 [None None None None None]
 [None None None None None]]
Lượt của bạn (X)
Dòng: 0
Cột: 1
[[None 'X' None None None]
 [None None 'O' None None]
 [None None 'X' None None]
 [None None None None None]
 [None None None None None]]
AI (O) đang tính toán...
[[None 'X' None None None]
 [None None 'O' None None]
 [None 'O' 'X' None None]
 [None None None None None]
 [None None None None None]]
Lượt của bạn (X)
Dòng: 4
Cột: 0
[[None 'X' None None None]
 [None None 'O' None None]
 [None 'O' 'X' None None]
 [None None None None No

In [5]:
import os
import math

# --- CẤU HÌNH CHO NXN ---
N = 5             # Kích thước bàn cờ (5x5, 10x10,...)
WIN_LEN = 4       # Số quân liên tiếp để thắng (Thường là 4 với bàn 5x5, 5 với bàn 10x10)
MAX_DEPTH = 3     # Độ sâu AI tìm kiếm (Tăng lên nếu muốn AI thông minh hơn)

def GetWinner(board):
    """
    Kiểm tra người chiến thắng trên bàn cờ NxN.
    """
    for i in range(N * N):
        if not isinstance(board[i], str): continue # Ô trống

        row, col = i // N, i % N
        curr = board[i]

        # Kiểm tra 4 hướng: Ngang, Dọc, Chéo xuôi, Chéo ngược
        for dr, dc in [(0, 1), (1, 0), (1, 1), (1, -1)]:
            count = 0
            for k in range(WIN_LEN):
                r, c = row + dr * k, col + dc * k
                if 0 <= r < N and 0 <= c < N and board[r * N + c] == curr:
                    count += 1
                else:
                    break
            if count == WIN_LEN:
                return curr
    return None

def PrintBoard(board):
    """
    In bàn cờ NxN theo định dạng lưới.
    """
    os.system('cls' if os.name == 'nt' else 'clear')
    print("-" * (N * 4 + 1))
    for i in range(N):
        row_str = "| "
        for j in range(N):
            cell = board[i * N + j]
            # Căn chỉnh để số có 2 chữ số không làm lệch hàng
            cell_display = cell if isinstance(cell, str) else str(cell).zfill(2)
            row_str += cell_display + " | "
        print(row_str)
        print("-" * (N * 4 + 1))

def GetAvailableCells(board):
    """
    Tối ưu cho NxN: Chỉ trả về các ô trống nằm gần các quân đã đánh.
    Nếu bàn cờ trống, trả về ô giữa.
    """
    occupied = [i for i, cell in enumerate(board) if isinstance(cell, str)]
    if not occupied:
        return [ (N * N) // 2 ]

    available = set()
    for idx in occupied:
        r, c = idx // N, idx % N
        for dr in range(-1, 2):
            for dc in range(-1, 2):
                nr, nc = r + dr, c + dc
                if 0 <= nr < N and 0 <= nc < N:
                    target_idx = nr * N + nc
                    if not isinstance(board[target_idx], str):
                        available.add(target_idx + 1) # +1 để khớp với logic cell - 1
    return list(available)

def EvaluateBoard(board):
    """
    Hàm lượng giá khi chưa kết thúc trận đấu (Heuristic).
    """
    winner = GetWinner(board)
    if winner == "X": return 100
    if winner == "O": return -100
    return 0

def minimax(position, depth, alpha, beta, isMaximizing):
    """
    Thuật toán Minimax kết hợp cắt tỉa Alpha-Beta và giới hạn độ sâu.
    """
    winner = GetWinner(position)
    if winner != None:
        return 100 - depth if winner == "X" else -100 + depth

    available = GetAvailableCells(position)
    if len(available) == 0 or depth >= MAX_DEPTH:
        return EvaluateBoard(position)

    if isMaximizing:
        maxEval = -math.inf
        for cell in available:
            position[cell - 1] = "X"
            eval = minimax(position, depth + 1, alpha, beta, False)
            position[cell - 1] = cell
            maxEval = max(maxEval, eval)
            alpha = max(alpha, eval)
            if beta <= alpha: break
        return maxEval
    else:
        minEval = math.inf
        for cell in available:
            position[cell - 1] = "O"
            eval = minimax(position, depth + 1, alpha, beta, True)
            position[cell - 1] = cell
            minEval = min(minEval, eval)
            beta = min(beta, eval)
            if beta <= alpha: break
        return minEval

def FindBestMove(currentPosition, AI):
    """
    Tìm nước đi tốt nhất cho AI.
    """
    bestVal = -math.inf if AI == "X" else math.inf
    bestMove = -1
    available = GetAvailableCells(currentPosition)

    for cell in available:
        currentPosition[cell - 1] = AI
        # Gọi minimax với độ sâu bắt đầu là 0
        moveVal = minimax(currentPosition, 0, -math.inf, math.inf, False if AI == "X" else True)
        currentPosition[cell - 1] = cell

        if (AI == "X" and moveVal > bestVal):
            bestVal = moveVal
            bestMove = cell
        elif (AI == "O" and moveVal < bestVal):
            bestVal = moveVal
            bestMove = cell

    return bestMove if bestMove != -1 else available[0]

def main():
    player = input(f"Ban muon choi X hay O (Ban co {N}x{N}, thang khi co {WIN_LEN} quan)? ").strip().upper()
    AI = "O" if player == "X" else "X"
    currentGame = [*range(1, N * N + 1)]
    currentTurn = "X" # X luon di truoc

    while True:
        if currentTurn == AI:
            print("AI dang tinh toan...")
            cell = FindBestMove(currentGame, AI)
            currentGame[cell - 1] = AI
            currentTurn = player
        else:
            PrintBoard(currentGame)
            while True:
                try:
                    humanInput = int(input(f"Luot ban ({player}), nhap so o: ").strip())
                    if humanInput in currentGame and not isinstance(currentGame[humanInput-1], str):
                        currentGame[humanInput - 1] = player
                        currentTurn = AI
                        break
                    else:
                        print("O khong hop le hoac da co nguoi danh!")
                except:
                    print("Vui long nhap so!")

        winner = GetWinner(currentGame)
        if winner != None:
            PrintBoard(currentGame)
            print(f"{winner} THANG!!!")
            break

        if all(isinstance(c, str) for c in currentGame):
            PrintBoard(currentGame)
            print("HOA!")
            break

if __name__ == "__main__":
    main()

Ban muon choi X hay O (Ban co 5x5, thang khi co 4 quan)? X
---------------------
| 01 | 02 | 03 | 04 | 05 | 
---------------------
| 06 | 07 | 08 | 09 | 10 | 
---------------------
| 11 | 12 | 13 | 14 | 15 | 
---------------------
| 16 | 17 | 18 | 19 | 20 | 
---------------------
| 21 | 22 | 23 | 24 | 25 | 
---------------------
Luot ban (X), nhap so o: 13
AI dang tinh toan...
---------------------
| 01 | 02 | 03 | 04 | 05 | 
---------------------
| 06 | O | 08 | 09 | 10 | 
---------------------
| 11 | 12 | X | 14 | 15 | 
---------------------
| 16 | 17 | 18 | 19 | 20 | 
---------------------
| 21 | 22 | 23 | 24 | 25 | 
---------------------
Luot ban (X), nhap so o: 12
AI dang tinh toan...
---------------------
| 01 | 02 | 03 | 04 | 05 | 
---------------------
| 06 | O | 08 | 09 | 10 | 
---------------------
| O | X | X | 14 | 15 | 
---------------------
| 16 | 17 | 18 | 19 | 20 | 
---------------------
| 21 | 22 | 23 | 24 | 25 | 
---------------------
Luot ban (X), nhap so o: 8
AI dan